In [ ]:
!git clone https://github.com/pzarghami/TML.git
%cd TML
!git submodule update --init --recursive
%cd assignment1/tml26_task1/

In [ ]:
import os
import sys
import torch
import pandas as pd
import requests
import random
import argparse

from pathlib import Path
from torch.utils.data import Dataset
from torchvision.models import resnet18
import torchvision.transforms as transforms


# config
BASE = Path.cwd()
PUB_PATH = BASE / "pub.pt"
PRIV_PATH = BASE / "priv.pt"
MODEL_PATH = BASE / "model.pt"
OUTPUT_CSV = BASE / "submission.csv"

BASE_URL = "http://34.63.153.158"   #DONOT CHANGE
API_KEY = "5930991bac7bfab215bc24486e4a5c0e"
TASK_ID = "01-mia"  #DONOT CHANGE



# dataset classes
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        label = self.labels[index]
        return id_, img, label

    def __len__(self):
        return len(self.ids)


class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]


# load datasets
print("Loading datasets...")
pub_ds = torch.load(PUB_PATH, weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)


# normalization (same as training)
MEAN = [0.7406, 0.5331, 0.7059]
STD = [0.1491, 0.1864, 0.1301]

transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Normalize(mean=MEAN, std=STD),
])

pub_ds.transform = transform
priv_ds.transform = transform


# load model
print("Loading model...")
model = resnet18(weights=None)
model.conv1 = torch.nn.Conv2d(3, 64, 3, 1, 1, bias=False)
model.maxpool = torch.nn.Identity()
model.fc = torch.nn.Linear(512, 9)

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import random
import time
import gc

from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18
import torchvision.transforms as T
from sklearn.metrics import roc_curve


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 2032
N_SHADOW = 8
EPOCHS = 65
LR = 0.1
WD = 1e-3
LS = 0.03
BATCH = 128
SUBSET = 0.5


MEAN = [0.7406, 0.5331, 0.7059]
STD  = [0.1491, 0.1864, 0.1301]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("device:", device)
print(f"shadow config: lr={LR} ep={EPOCHS} wd={WD} ls={LS} n={N_SHADOW}")


tf_eval = T.Compose([
    T.Resize(32),
    T.Normalize(mean=MEAN, std=STD),
])

tf_train = T.Compose([
    T.Resize(32),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.Normalize(mean=MEAN, std=STD),
])


def build_resnet():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(512, 9)
    return m


def tpr_at_5(gt, scores):
    fpr, tpr, _ = roc_curve(gt, scores)
    return tpr[np.argmin(np.abs(fpr - 0.05))]


def per_class_zscore(scores, labels):
    out = np.zeros_like(scores, dtype=np.float64)
    for c in np.unique(labels):
        m = labels == c
        v = scores[m]
        sd = v.std()
        if sd > 0:
            out[m] = (v - v.mean()) / sd
    return out


def to01(x):
    a, b = x.min(), x.max()
    return (x - a) / (b - a) if b > a else np.full_like(x, 0.5)


def true_class_conf(model, ds):
    ds.transform = tf_eval
    loader = DataLoader(ds, batch_size=256, shuffle=False)
    out = []
    with torch.no_grad():
        for batch in loader:
            imgs = batch[1].to(device)
            labels = batch[2].to(device)
            probs = torch.softmax(model(imgs), dim=1)
            out.append(probs[range(len(labels)), labels].cpu().numpy())
    return np.concatenate(out)


print("loading data...")
pub_ds = torch.load(PUB_PATH, weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)
priv_ds.membership = [0] * len(priv_ds.ids)

gt = np.array(list(pub_ds.membership))
pub_y = np.array(list(pub_ds.labels))
priv_y = np.array(list(priv_ds.labels))

N_pub = len(pub_ds.ids)
N_priv = len(priv_ds.ids)
print(f"public: {N_pub}, private: {N_priv}")


# target
print("loading target model...")
target = build_resnet()
target.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
target = target.to(device).eval()

t_conf_pub = true_class_conf(target, pub_ds)
t_conf_priv = true_class_conf(target, priv_ds)
print(f"target mean conf on public: {t_conf_pub.mean():.4f}")

del target
torch.cuda.empty_cache()


# train shadow models
print(f"\ntraining {N_SHADOW} shadow models\n")

shadow_pub = []
shadow_priv = []
shadow_masks = []

for s in range(N_SHADOW):
    t0 = time.time()

    # random 50% subset
    perm = np.random.permutation(N_pub)
    train_idx = perm[:int(N_pub * SUBSET)]
    mask = np.zeros(N_pub, dtype=bool)
    mask[train_idx] = True
    shadow_masks.append(mask)

    model = build_resnet().to(device)
    opt = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WD)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    pub_ds.transform = tf_train
    train_loader = DataLoader(Subset(pub_ds, train_idx), batch_size=BATCH,
                              shuffle=True, drop_last=True, num_workers=2)

    model.train()
    for ep in range(EPOCHS):
        for batch in train_loader:
            x = batch[1].to(device)
            y = batch[2].to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y, label_smoothing=LS)
            loss.backward()
            opt.step()
        sched.step()

    model.eval()
    pc = true_class_conf(model, pub_ds)
    qc = true_class_conf(model, priv_ds)
    shadow_pub.append(pc)
    shadow_priv.append(qc)

    print(f"  shadow {s+1}/{N_SHADOW}: {time.time()-t0:.0f}s, pub conf {pc.mean():.4f}")

    del model, opt, sched
    torch.cuda.empty_cache()
    gc.collect()


# rmia computations
print("\ncomputing RMIA scores")

shadow_pub_arr = np.stack(shadow_pub)       
shadow_priv_arr = np.stack(shadow_priv)     
shadow_mask_arr = np.stack(shadow_masks) 

out_mask = ~shadow_mask_arr
out_counts = out_mask.sum(axis=0)

avg_pub = np.zeros(N_pub)
has_out = out_counts > 0
avg_pub[has_out] = (shadow_pub_arr[:, has_out] * out_mask[:, has_out]).sum(axis=0) / out_counts[has_out]

avg_pub[~has_out] = shadow_pub_arr[:, ~has_out].mean(axis=0)

avg_priv = shadow_priv_arr.mean(axis=0)

eps = 1e-10
log_pub = np.log(t_conf_pub + eps) - np.log(avg_pub + eps)
log_priv = np.log(t_conf_priv + eps) - np.log(avg_priv + eps)

print(f"log score: members mean = {log_pub[gt==1].mean():.4f}, "
      f"non-members mean = {log_pub[gt==0].mean():.4f}, "
      f"diff = {log_pub[gt==1].mean() - log_pub[gt==0].mean():.4f}")


def rmia_pairwise(target_logs, ref_logs, target_labels, ref_labels,
                  gamma=1.0, same_class=True):
    out = np.zeros(len(target_logs))
    log_g = np.log(gamma + 1e-10)

    if same_class:
        sorted_per_class = {c: np.sort(ref_logs[ref_labels == c])
                            for c in np.unique(ref_labels)}
        for c in np.unique(target_labels):
            sel = target_labels == c
            refs = sorted_per_class.get(c)
            if refs is None or len(refs) == 0:
                continue
            thr = target_logs[sel] - log_g
            out[sel] = np.searchsorted(refs, thr, side="left") / len(refs)
    else:
        refs = np.sort(ref_logs)
        thr = target_logs - log_g
        out = np.searchsorted(refs, thr, side="left") / len(refs)

    return out


# search gamma
print("\ngamma search")
gammas = [0.6, 0.7, 0.8, 0.9, 1.0, 1.15, 1.3, 1.5, 2.0]

rmia_pub = {}
rmia_priv = {}

for g in gammas:
    for sc in (True, False):
        tag = f"g={g} {'same' if sc else 'all'}"
        rmia_pub[tag] = rmia_pairwise(log_pub, log_pub, pub_y, pub_y, g, sc)
        rmia_priv[tag] = rmia_pairwise(log_priv, log_pub, priv_y, pub_y, g, sc)


print("\n(TPR@5%FPR)\n")

candidates = {}

candidates["log_score raw"] = (log_pub, log_priv)
candidates["log_score norm"] = (per_class_zscore(log_pub, pub_y),
                                per_class_zscore(log_priv, priv_y))
candidates["target_conf raw"] = (t_conf_pub, t_conf_priv)
candidates["target_conf norm"] = (per_class_zscore(t_conf_pub, pub_y),
                                  per_class_zscore(t_conf_priv, priv_y))

for tag in rmia_pub:
    candidates[tag + " raw"] = (rmia_pub[tag], rmia_priv[tag])
    candidates[tag + " norm"] = (per_class_zscore(rmia_pub[tag], pub_y),
                                 per_class_zscore(rmia_priv[tag], priv_y))

best_name = None
best_tpr = 0

for name, (pub_s, _) in candidates.items():
    score = tpr_at_5(gt, pub_s)
    flag = "best" if score > best_tpr else ""
    if score > best_tpr:
        best_tpr = score
        best_name = name
    print(f"  {name:<28} {score:.4f}{flag}")

print(f"\nbest: {best_name} : TPR@5%FPR = {best_tpr:.4f}")


final = to01(candidates[best_name][1])
priv_ids = list(priv_ds.ids)
pd.DataFrame({"id": priv_ids, "score": final}).to_csv(OUTPUT_CSV, index=False)

print(f"\nsaved {OUTPUT_CSV} ({len(priv_ids)} rows)")
print(f"method: {best_name}")
print(f"score range: [{final.min():.4f}, {final.max():.4f}]")